# **Carga de datos**

In [1]:
import pandas as pd
import numpy as np
from prophet import Prophet
import warnings
warnings.filterwarnings('ignore')

PATH = '/kaggle/input/competitions/store-sales-time-series-forecasting/'

train = pd.read_csv(PATH + 'train.csv')
train['date'] = pd.to_datetime(train['date'])

test = pd.read_csv(PATH + 'test.csv')
test['date'] = pd.to_datetime(test['date'])

oil      = pd.read_csv(PATH + 'oil.csv');      oil['date'] = pd.to_datetime(oil['date'])
holidays = pd.read_csv(PATH + 'holidays_events.csv'); holidays['date'] = pd.to_datetime(holidays['date'])
stores   = pd.read_csv(PATH + 'stores.csv')
sample   = pd.read_csv(PATH + 'sample_submission.csv')

print("Train:", train.shape)
print("Test:", test.shape)
print("IDs a predecir:", len(sample))

Train: (3000888, 6)
Test: (28512, 5)
IDs a predecir: 28512


# **Preprocesamiento**

In [2]:
# Unir con precio del petróleo (impacta en el consumo)
train = train.merge(oil, on='date', how='left')
test  = test.merge(oil, on='date', how='left')

# Rellenar NaN del petróleo
train['dcoilwtico'] = train['dcoilwtico'].interpolate()
test['dcoilwtico']  = test['dcoilwtico'].fillna(method='ffill')

# Preparar feriados para Prophet
holidays_prophet = holidays[
    holidays['locale'] == 'National'
][['date', 'description']].copy()
holidays_prophet.columns = ['ds', 'holiday']
holidays_prophet = holidays_prophet.drop_duplicates(subset='ds')

print(f"Combinaciones tienda-familia: {test.groupby(['store_nbr','family']).ngroups}")

Combinaciones tienda-familia: 1782


# **Prophet**

In [3]:
predictions = []

combos = test[['store_nbr', 'family']].drop_duplicates().values
total = len(combos)
print(f"Entrenando {total} modelos...\n")

for i, (store, family) in enumerate(combos):
    if i % 100 == 0:
        print(f"Progreso: {i}/{total} ({round(i/total*100)}%)")

    # Datos de entrenamiento para esta combinación
    mask_train = (train['store_nbr'] == store) & (train['family'] == family)
    df_train = train[mask_train][['date', 'sales', 'onpromotion', 'dcoilwtico']].copy()
    df_train.columns = ['ds', 'y', 'onpromotion', 'oil']
    df_train['y'] = df_train['y'].clip(lower=0)

    # Datos de test para esta combinación
    mask_test = (test['store_nbr'] == store) & (test['family'] == family)
    df_test = test[mask_test][['date', 'id', 'onpromotion', 'dcoilwtico']].copy()
    df_test.columns = ['ds', 'id', 'onpromotion', 'oil']

    # Si no hay datos suficientes, predecir 0
    if len(df_train) < 10:
        df_test['sales'] = 0
        predictions.append(df_test[['id', 'sales']])
        continue

    try:
        m = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
            holidays=holidays_prophet,
            seasonality_mode='multiplicative'
        )
        m.add_regressor('onpromotion')
        m.add_regressor('oil')
        m.fit(df_train)

        forecast = m.predict(df_test[['ds', 'onpromotion', 'oil']])
        df_test['sales'] = forecast['yhat'].clip(lower=0).values

    except Exception as e:
        df_test['sales'] = df_train['y'].mean()

    predictions.append(df_test[['id', 'sales']])

print("\n finalizado!")

Entrenando 1782 modelos...

Progreso: 0/1782 (0%)
Progreso: 100/1782 (6%)
Progreso: 200/1782 (11%)
Progreso: 300/1782 (17%)
Progreso: 400/1782 (22%)
Progreso: 500/1782 (28%)
Progreso: 600/1782 (34%)
Progreso: 700/1782 (39%)
Progreso: 800/1782 (45%)
Progreso: 900/1782 (51%)
Progreso: 1000/1782 (56%)
Progreso: 1100/1782 (62%)
Progreso: 1200/1782 (67%)
Progreso: 1300/1782 (73%)
Progreso: 1400/1782 (79%)
Progreso: 1500/1782 (84%)
Progreso: 1600/1782 (90%)
Progreso: 1700/1782 (95%)

 finalizado!


# **Generar submission.csv**

In [4]:
submission = pd.concat(predictions).sort_values('id').reset_index(drop=True)
submission['sales'] = submission['sales'].clip(lower=0).round(4)

submission.to_csv('/kaggle/working/submission.csv', index=False)

print(f"Filas: {len(submission)}")
print(f"Sales negativas: {(submission['sales'] < 0).sum()}")
print("\nPrimeras filas:")
print(submission.head())

Filas: 28512
Sales negativas: 0

Primeras filas:
        id      sales
0  3000888     3.2512
1  3000889     0.0000
2  3000890     2.4086
3  3000891  1587.7488
4  3000892     0.1253


# **Validación**

In [5]:
sample = pd.read_csv(PATH + 'sample_submission.csv')

assert len(submission) == len(sample)
assert list(submission.columns) == ['id', 'sales']
assert submission['id'].equals(sample['id']) 

print("listo")

listo


# **Conslusión**
## 🇦🇷 Español

Se entrenaron **1,782 modelos Prophet** — uno por cada combinación tienda-familia — 
utilizando como variables externas el precio del petróleo (`dcoilwtico`) y las 
promociones (`onpromotion`), junto con los feriados nacionales de Ecuador.

**Decisiones clave:**
- **Seasonality mode multiplicativo** — apropiado para series con tendencia creciente
- **Regressores externos** — precio del petróleo e indicador de promoción
- **Feriados nacionales** — incorporados explícitamente al modelo
- **Clip lower=0** — las ventas no pueden ser negativas (requerido por RMSLE)

**Limitaciones:**
- No se optimizaron hiperparámetros 
- No se incorporaron features de tienda 

### 🇺🇸 English

**1,782 Prophet models** were trained — one per store-family combination — 
using oil price (`dcoilwtico`), promotions (`onpromotion`), and Ecuadorian 
national holidays as external variables.

**Key decisions:**
- **Multiplicative seasonality** — appropriate for series with growing trend
- **External regressors** — oil price and promotion indicator
- **National holidays** — explicitly incorporated into the model
- **Clip lower=0** — sales cannot be negative (required by RMSLE)

**Limitations:**
- No hyperparameter optimization (Optuna/grid search)
- Store features not incorporated (city, type, cluster)
- Training time: ~2 hours for all 1,782 series
